# 00: Connection check

Confirms that the kernel can reach the data lake

**Before running**, in a terminal:

```powershell
aws sso login --profile ciccada
```


## Setup

Find the repository root by walking upwards, then put it on `sys.path`, so the
imports work regardless of where Jupyter was started.


In [ ]:
# Reload edited .py modules without restarting the kernel.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from bms_sa_review.ami_data_analysis.config import ami_config as C
from bms_sa_review.ami_data_analysis.lib import ami_athena as A
from bms_sa_review.ami_data_analysis.lib import ami_diagnostics as D

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

A.reset_scan_log()

print("Repository root:", REPO_ROOT)
print("Store dir      :", C.STORE_DIR, "(created in Phase 4; absence here is expected)")
print("Databases      :", C.SA, "|", C.SAI, "|", C.BOM_DB)


## 1. Environment

In [ ]:
env = D.environment_report()
display(env)

missing = env.loc[env["required"] & (env["status"] == "MISSING")]
if len(missing):
    raise RuntimeError(
        "Required packages missing:\n"
        + missing[["item", "detail"]].to_string(index=False)
    )
print("Environment OK.")


## 2. Credentials

Check for expired SSO token

In [ ]:
status = A.credential_status()

if status["ok"]:
    print("SSO session valid.")
#    print(f"  profile : {status['profile']}")
#    print(f"  region  : {status['region']}")
#    print(f"  account : {status['account']}")
#    print(f"  identity: {status['arn']}")
else:
    print("SSO SESSION NOT USABLE")
    print(f"  reason : {status['reason']}")
    print(f"  profile: {status['profile']}   region: {status['region']}")
    print()
    print(status["remedy"])

aws_checks = D.aws_report(status)
# display(aws_checks)


In [ ]:
# Stop here if the session is dead. Everything below needs it.
# A.require_credentials()

## 3. Both Glue databases are reachable

- `solar_analytics` is the legacy Hive catalogue
- `solar_analytics_iceberg` is the primary one 

In [ ]:
cfg = A.get_aws_config()

all_databases = cfg.databases()
display(all_databases)

reachable = {}
for db in (C.SA, C.SAI, C.BOM_DB):
    try:
        listed = cfg.tables(db)
        reachable[db] = len(listed)
        print(f"{db:<28} {len(listed):>4} tables")
    except Exception as exc:
        reachable[db] = None
        print(f"{db:<28} UNREACHABLE -- {type(exc).__name__}: {exc}")


## 4. Trivial query run


In [ ]:
trivial = A.aq("SELECT 1 AS ok", database=C.SAI, label="SELECT 1")
display(trivial)

dim_sample = A.aq("SELECT * FROM circuits LIMIT 5", database=C.SA, label="circuits LIMIT 5")
display(dim_sample)


## 5. The partition guard

`ami_athena.aq()` refuses to run an unpartitioned query against `ts` or the other large tables. 
Nothing here runs a query.


In [ ]:
guard_demo = pd.DataFrame([
    {"sql": sql, "blocked": bool(A.check_partition_filters(sql))}
    for sql in [
        "SELECT * FROM ts",
        "SELECT count(*) FROM ts",
        "SELECT * FROM ts WHERE year = 2025 AND month = 1 LIMIT 5",
        'SELECT * FROM "ts$partitions"',
        "SELECT * FROM circuits",
    ]
])
display(guard_demo)

assert guard_demo.blocked.tolist() == [True, True, False, False, False], (
    "The partition guard is not behaving as expected -- do not proceed to 01."
)
print("Partition guard armed.")


## 6. Conventions, and what is still unresolved

Every methodological choice this package depends on, with its resolution state.

At Phase 0 almost everything is `resolved = False`. That is the point: these are
declared placeholders awaiting evidence from notebooks 02 and 03, not silent
defaults. `manifest()` will print this same table alongside every result from
Phase 5, so an unresolved choice cannot quietly become a decision.


In [ ]:
display(C.describe_conventions())

pending = C.unresolved()
print(f"{len(pending)} convention(s) still unresolved:")
for name in pending:
    print("  -", name)


## 7. Verdict


In [ ]:
checks = pd.concat([
    aws_checks,
    pd.DataFrame([
        D.check("Glue: solar_analytics reachable", "listed",
                reachable.get(C.SA), reachable.get(C.SA) is not None, group="catalog"),
        D.check("Glue: solar_analytics_iceberg reachable", "listed",
                reachable.get(C.SAI), reachable.get(C.SAI) is not None, group="catalog"),
        D.check("Athena executes", 1,
                None if trivial.empty else int(trivial.ok.iloc[0]),
                (not trivial.empty) and int(trivial.ok.iloc[0]) == 1, group="athena"),
        D.check("dimension read returns rows", "5 rows", f"{len(dim_sample)} rows",
                len(dim_sample) == 5, group="athena"),
    ]),
], ignore_index=True)

display(checks)
assert D.summarise(checks, label="00 connection check"), (
    "Connection check failed -- fix the above before running 01."
)


## 8. Cost check

`source = unavailable` means the scan figure could not be recovered from the
Athena response, **not** that the query was free. The total is a lower bound in
that case.


In [ ]:
display(A.scan_report())
